** Data Preprocessing**

*My Steps -> Summary*
- Merge: all 3 CSVs
- Keep: useful columns
- Remove: missing values
- Create: sentiment labels
- Clean: review text
- Remove: very short reviews
- Save: final dataset

In [10]:
#Imports

import pandas as pd
from pathlib import Path
import re

In [ ]:
#Merging the csv data
data_folder = Path("../data")
csv_files = list(data_folder.glob("*.csv"))
csv_files

#Create DataFrame of Data
df_list = []

for file in csv_files:
    temp_df = pd.read_csv(file, low_memory=False)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

df.shape

(67992, 27)

In [ ]:
#Review data
df = df [[
    "name",
    "reviews.rating",
    "reviews.text"
]]

df.head()

#Remove rows where at least one of the three columns is empty
df = df.dropna(subset=["name", "reviews.rating", "reviews.text"])
df.shape

(61199, 3)

In [ ]:
#Convert the ratings to integers
df["reviews.rating"] = df["reviews.rating"].astype(int)
df["reviews.rating"].unique()

#Create sentiment column
def map_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"
    

df["sentiment"] = df["reviews.rating"].apply(map_sentiment)
df["sentiment"].value_counts()

#Outcome: Many positive reviews. Pretty imbalanced


#Clean the text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ",text) #remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", " ",text) # remove special characters, keep letters, numbers, spaces only.
    text = re.sub(r"\s+", " ",text).strip() #remove the extra spaces too

    return text


#apply funciton to every review
df["clean_review"] = df["reviews.text"].apply(clean_text) 

#show original vs cleaned text
df[["reviews.text", "clean_review"]].head() 


#show full text side by side (no truncation)
pd.set_option('display.max_colwidth', None)
df[["reviews.text", "clean_review"]].sample(10)

#remove short reviews like "good", "ok", etc.
df = df[df["clean_review"].str.len() > 10]
df.shape

(59900, 5)

In [17]:
#Save the cleaned dataset

#define path for cleaned file to be saved to
output_path= Path("../outputs/clean_reviews.csv")

#save the df as a csv
df.to_csv(output_path, index=False)

#print the condirmation for double-check
print("Saved to:", output_path)

Saved to: ../outputs/clean_reviews.csv
